In [2]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
df = pd.read_csv("processed.cleveland.data", header=None, names=columns, na_values="?")

categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
X = df.drop(columns="num")
# X=df[categorical_features]
# y=df["num"]
y = (df["num"] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# preprocessor = ColumnTransformer(
#     [("categorical", make_pipeline(
#         SimpleImputer(strategy="most_frequent"),
#         OneHotEncoder(handle_unknown="ignore")
#     ), categorical_features)]
# )

preprocessor1=SimpleImputer(strategy="most_frequent")
preprocessor2=OneHotEncoder(handle_unknown="ignore")
X_train_imputed = preprocessor1.fit_transform(X_train)
X_train_encoded = preprocessor2.fit_transform(X_train_imputed)
X_test_imputed = preprocessor1.transform(X_test)
X_test_encoded = preprocessor2.transform(X_test_imputed)


In [1]:
print("kernel is alive")

kernel is alive


In [ ]:
clf_gini = DecisionTreeClassifier(criterion='gini', random_state=42)
# fit the model
clf_gini.fit(X_train_encoded, y_train)
y_pred_gini = clf_gini.predict(X_test_encoded)
print('Model accuracy score with criterion gini index: {0:0.4f}'. format(accuracy_score(y_test, y_pred_gini)))
y_pred_train_gini = clf_gini.predict(X_train_encoded)
print('Training-set accuracy score: {0:0.4f}'. format(accuracy_score(y_train, y_pred_train_gini)))


In [ ]:
clf_en = DecisionTreeClassifier(criterion='entropy',random_state=42)
# fit the model
clf_en.fit(X_train_encoded, y_train)
y_pred_en = clf_en.predict(X_test_encoded)
print('Model accuracy score with criterion entropy: {0:0.4f}'. format(accuracy_score(y_test, y_pred_en)))
y_pred_train_en = clf_en.predict(X_train_encoded)
print('Training-set accuracy score: {0:0.4f}'. format(accuracy_score(y_train, y_pred_train_en)))

In [ ]:

param={
    "min_samples_split": [2, 5, 10, 20],
    "max_features": [None, "sqrt", "log2"]
}

grid = GridSearchCV(
    estimator=clf_gini,
    param_grid=param,
    scoring="accuracy",
    cv=5
)

grid.fit(X_train_encoded, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validation accuracy:", grid.best_score_)

In [ ]:
forest = RandomForestClassifier(random_state=42)

param_grid_rf = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10, 20]
}

grid_rf = GridSearchCV(
    estimator=forest,
    param_grid=param_grid_rf,
    scoring="accuracy",
    cv=5
)

grid_rf.fit(X_train_encoded, y_train)

print("Best parameters:", grid_rf.best_params_)
print("Best cross-validation accuracy:", grid_rf.best_score_)

best_forest = grid_rf.best_estimator_
y_pred_rf = best_forest.predict(X_test_encoded)

print("Test accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, zero_division=0))